# Laboratorio 2. El tablero no cuadra

Alguien dejó un pipeline que dice éxito. Finanzas dice que las ventas no cuadran con el archivo.

Su trabajo no es seguir una receta. Es dejar `oro_fact_ventas` usable y explicar qué apartaron.

Reglas del negocio, nada más:

- Una fila por `venta_id`.
- Solo entran las tiendas 1, 2 y 3.
- `monto` tiene que ser número. Si viene con coma, se interpreta. Si no se puede, se aparta.
- Una columna que no estaba en el contrato no entra al hecho.
- Lo apartado queda en `cuarentena_ventas`, con un motivo.
- Correr el notebook otra vez no duplica el hecho.

No editen el CSV. Si una prueba falla, el hallazgo es de ustedes.

In [0]:
dbutils.widgets.text("ruta_csv", "file:/Workspace/Users/202401441@ujmd.edu.sv/ventas_export.csv")
ruta = dbutils.widgets.get("ruta_csv")

bronce = spark.read.option("header", True).option("inferSchema", False).csv(ruta)
bronce.createOrReplaceTempView("bronce_ventas")

spark.sql("""
create or replace temp view dim_tienda as
select * from values
  (1, 'Centro'),
  (2, 'Escalon'),
  (3, 'Soyapango')
as t(tienda_id, nombre)
""")

print("Filas en el archivo:", bronce.count())
print("Columnas:", bronce.columns)

Filas en el archivo: 59
Columnas: ['venta_id', 'fecha', 'tienda_id', 'producto_id', 'cantidad', 'monto', 'medio_pago', 'canal']


## El pipeline que heredaron

Esta celda es el flujo roto. Léanla, no la copien como entrega. Carga todo, no aparta nada y si la corren dos veces duplica.

In [0]:
spark.sql("""
create or replace temp view oro_roto as
select * from bronce_ventas
""")
display(spark.sql("select count(*) as filas_oro_roto from oro_roto"))

## Su pipeline

Reemplacen esta celda. Tiene que crear `oro_fact_ventas` y `cuarentena_ventas`.

`oro_fact_ventas`: `venta_id`, `fecha`, `tienda_id`, `producto_id`, `cantidad`, `monto`, `medio_pago`.

`cuarentena_ventas`: las que no entran, con una columna `motivo`.

In [0]:
ruta = dbutils.widgets.get("ruta_csv")

bronce = spark.read.option("header", True).option("inferSchema", False).csv(ruta)
bronce.createOrReplaceTempView("bronce_eventas")

spark.sql("""
CREATE OR REPLACE TEMP VIEW ventas_preparadas AS
SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto,
    medio_pago,
    TRY_CAST(tienda_id AS INT) AS tienda_id_num,
    TRY_CAST(REPLACE(TRIM(monto), ',', '.') AS DOUBLE) AS monto_num,
    ROW_NUMBER() OVER (
        PARTITION BY venta_id
        ORDER BY fecha, tienda_id, producto_id, cantidad, monto, medio_pago
    ) AS rn
FROM bronce_eventas
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW oro_fact_ventas AS
SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto_num AS monto,
    medio_pago
FROM ventas_preparadas
WHERE rn = 1
  AND tienda_id_num IN (1, 2, 3)
  AND monto_num IS NOT NULL
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW cuarentena_ventas AS
SELECT
    venta_id,
    fecha,
    tienda_id,
    producto_id,
    cantidad,
    monto,
    medio_pago,
    CASE
        WHEN rn > 1 THEN 'venta_id duplicado'
        WHEN tienda_id_num NOT IN (1, 2, 3) OR tienda_id_num IS NULL
            THEN 'tienda fuera del contrato'
        WHEN monto_num IS NULL
            THEN 'monto no numerico'
        ELSE 'registro no valido'
    END AS motivo
FROM ventas_preparadas
WHERE rn > 1
   OR tienda_id_num NOT IN (1, 2, 3)
   OR tienda_id_num IS NULL
   OR monto_num IS NULL
""")

display(spark.sql("SELECT * FROM oro_fact_ventas"))

venta_id,fecha,tienda_id,producto_id,cantidad,monto,medio_pago
1001,2026-10-01,1,501,2,18.5,efectivo
1002,2026-10-01,1,502,1,7.25,tarjeta
1004,2026-10-01,2,504,1,12.5,efectivo
1005,2026-10-01,3,505,4,31.0,efectivo
1007,2026-10-02,1,507,2,15.0,efectivo
1042,2026-10-02,1,501,1,18.5,efectivo
1401,2026-10-04,1,540,1,5.5,efectivo
2001,2026-10-01,1,501,1,4.0,efectivo
3001,2026-10-01,1,510,1,6.75,efectivo
3002,2026-10-01,1,511,2,9.0,tarjeta


## Pruebas

No dicen cómo arreglarlo. Dicen si ya quedó. Las seis tienen que decir PASÓ.

In [0]:
oro = spark.table("oro_fact_ventas")
cuarentena = spark.table("cuarentena_ventas")
bronce = spark.table("bronce_ventas")

def prueba(nombre, ok):
    print(("PASÓ  " if ok else "FALLÓ ") + nombre)
    return ok

duplicados = spark.sql("""
select count(*) c from (
  select venta_id from oro_fact_ventas
  group by venta_id having count(*) > 1
)""").first()[0]

tiendas_malas = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(tienda_id as int) not in (1, 2, 3)
""").first()[0]

montos_malos = spark.sql("""
select count(*) c from oro_fact_ventas
where try_cast(monto as double) is null
""").first()[0]

columnas_de_mas = [c for c in oro.columns if c.lower() == "canal"]
sin_motivo = "motivo" not in [c.lower() for c in cuarentena.columns]
oro_inflado = oro.count() >= bronce.count()
cuarentena_vacia = cuarentena.count() == 0

ok = []
ok.append(prueba("una fila por venta_id", duplicados == 0))
ok.append(prueba("solo tiendas 1, 2 y 3", tiendas_malas == 0))
ok.append(prueba("monto numérico en el hecho", montos_malos == 0))
ok.append(prueba("canal no entra al hecho", len(columnas_de_mas) == 0))
ok.append(prueba("cuarentena con motivo y con filas", (not sin_motivo) and (not cuarentena_vacia)))
ok.append(prueba("oro menor que el archivo", not oro_inflado))
print("-")
print("LISTO" if all(ok) else "SIGAN. Hay pruebas en rojo.")

PASÓ  una fila por venta_id
PASÓ  solo tiendas 1, 2 y 3
PASÓ  monto numérico en el hecho
PASÓ  canal no entra al hecho
PASÓ  cuarentena con motivo y con filas
PASÓ  oro menor que el archivo
-
LISTO


## Desafío o reto

Cuando las seis digan PASÓ, respondan en esta celda, sin mirar la guía:

1. ¿Qué pasó con la misma venta llegada dos veces?
en este paso se conservo una sola fila por ventas_id y la que se repitio se envio a cuarentena_ventas

2. ¿Qué hicieron con un monto que venía en texto?
en este caso se intento convertir a numero, se remplazò la coma por punto y si no se podia convertir se enviaba a cuarentena_ventas

3. ¿Por qué una tienda se quedó fuera?
Porque solo se permiten las tiendas con tienda_id 1, 2 y 3. Las demas se enviaron a cuarentena

4. ¿Dónde quedó la columna que no estaba en el contrato?
La columna canal quedo fuera de oro_fact_ventas, porque no forma parte de las columnas definidas para el hecho.

5. Si corren el notebook otra vez, ¿el hecho crece?
En este caso no, el pipeline reemplaza las vistas y mantiene una sola fila por venta_id entonces si lo vuelvo a ejecturar no se se duplica 


El que solo pega la consulta y no puede explicar una de estas, no termino el laboratorio